# Validação e fusão com dados sintéticos

Demonstração do núcleo do projeto. Não utiliza exames nem reproduz o score público E74.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

raiz = Path.cwd()
if not (raiz / 'src').exists():
    raiz = raiz.parent
sys.path.insert(0, str(raiz / 'src'))
from rsna_knee.data_contract import DataSchema
from rsna_knee.splits import build_group_split
from rsna_knee.metrics import macro_roc_auc
from rsna_knee.fusion import blend_predictions_by_logit
from rsna_knee.submission import validate_submission


## Separar grupos antes de avaliar

Dois estudos do mesmo paciente sintético ficam no mesmo lado da divisão.


In [ ]:
rng = np.random.default_rng(42)
rotulos = ('alvo_a', 'alvo_b')
verdade = rng.integers(0, 2, size=(240, 2))
frame = pd.DataFrame(verdade, columns=rotulos)
frame['study_id'] = ['sintetico_' + str(i) for i in range(len(frame))]
frame['patient_id'] = np.repeat(np.arange(120), 2)
schema = DataSchema(id_column='study_id', group_column='patient_id', label_columns=rotulos)
split = build_group_split(frame, schema, seed=42)
treino = frame.iloc[list(split.train_indices)]
validacao = frame.iloc[list(split.validation_indices)]
assert set(treino.patient_id).isdisjoint(validacao.patient_id)
print(f'{len(treino)} estudos de treino; {len(validacao)} de validação; sem grupos compartilhados.')


## Comparar previsões e fusão fixa

As previsões abaixo são simuladas a partir dos alvos apenas para ilustrar as funções; não são estimativas de generalização de um modelo treinado. O peso de fusão é fixo e não foi escolhido no conjunto avaliado.


In [ ]:
y = validacao[list(rotulos)].to_numpy()
logits_a = (2 * y - 1) * 0.8 + rng.normal(size=y.shape)
logits_b = (2 * y - 1) * 0.6 + rng.normal(size=y.shape)
prob_a = 1 / (1 + np.exp(-logits_a))
prob_b = 1 / (1 + np.exp(-logits_b))
fusao = blend_predictions_by_logit(prob_a, prob_b)
for nome, previsoes in [('Braço A', prob_a), ('Braço B', prob_b), ('Fusão fixa', fusao)]:
    score, por_alvo = macro_roc_auc(y, previsoes, rotulos)
    print(nome, round(score, 4), por_alvo)


## Validar o contrato de saída

O validador alinha IDs e colunas ao template e verifica probabilidades. Nenhum envio externo é feito.


In [ ]:
template = pd.DataFrame({'study_id': validacao.study_id.to_numpy(), 'alvo_a': 0., 'alvo_b': 0.})
predicoes = pd.DataFrame(fusao, columns=rotulos)
predicoes.insert(0, 'study_id', validacao.study_id.to_numpy())
validado = validate_submission(template, predicoes.iloc[::-1])
assert validado.study_id.tolist() == template.study_id.tolist()
print(f'{len(validado)} linhas validadas com a ordem do template.')
validado.head()
